# Overpass

In [1]:
import folium
from folium.plugins import MarkerCluster
import overpy

In [2]:
# List of known instances

# https://overpass-api.de/api/interpreter (Default)
# https://overpass.kumi.systems/api/interpreter (Works best)
# https://maps.mail.ru/osm/tools/overpass/api/interpreter

In [13]:
# Instantiating the Overpass API client with a specific URL
api = overpy.Overpass(url='https://overpass.kumi.systems/api/interpreter')

In [14]:
# Location: Bonn Hauptbahnhof
lon = 7.096739
lat = 50.732063

# Radius in meters
radius = 200

In [15]:
# Query to fetch nearby amenities and public transport stops
query = f"""
[out:json][timeout:60];
(
  node["public_transport"="stop_position"](around:{radius}, {lat}, {lon});
  node["shop"~"supermarket|pharmacy"](around:{radius}, {lat}, {lon});
  node["amenity"~"cafe|restaurant"](around:{radius}, {lat}, {lon});
  node["amenity"~"school|university|arts_centre"](around:{radius}, {lat}, {lon});
);
out body;
"""

In [16]:
# Execute query
try:
    result = api.query(query)
    print(f'Found {len(result.nodes)} nodes.')

except Exception as e:
    print(f'Error executing Overpass query: {e}')

Found 44 nodes.


In [17]:
print(f'Overpass API result: {result}')
print(f'Overpass API nodes: {result.nodes}')
print(f'First node: {result.nodes[0]}')
print(f'First node ID: {result.nodes[0].id}')
print(f'First node coordinates: {result.nodes[0].lat}, {result.nodes[0].lon}')
print(f'First node tags: {result.nodes[0].tags}')
print(f'First node name: {result.nodes[0].tags.keys()}')

Overpass API result: <overpy.Result object at 0x00000219EDDF55D0>
Overpass API nodes: [<overpy.Node id=31819742 lat=50.7317635 lon=7.0993974>, <overpy.Node id=86123030 lat=50.7320025 lon=7.0967372>, <overpy.Node id=253075334 lat=50.7325615 lon=7.0986046>, <overpy.Node id=293704285 lat=50.7329267 lon=7.0960932>, <overpy.Node id=324744562 lat=50.7319234 lon=7.0966881>, <overpy.Node id=517472749 lat=50.7320888 lon=7.0986878>, <overpy.Node id=523276761 lat=50.7319478 lon=7.0988917>, <overpy.Node id=571394730 lat=50.7331976 lon=7.0969030>, <overpy.Node id=571793858 lat=50.7334261 lon=7.0982747>, <overpy.Node id=1035438103 lat=50.7309111 lon=7.0966169>, <overpy.Node id=1074707003 lat=50.7318870 lon=7.0990683>, <overpy.Node id=1074707008 lat=50.7321454 lon=7.0990238>, <overpy.Node id=1074707029 lat=50.7318710 lon=7.0988958>, <overpy.Node id=1074707103 lat=50.7320172 lon=7.0991879>, <overpy.Node id=1074707178 lat=50.7319716 lon=7.0994149>, <overpy.Node id=1198542409 lat=50.7321232 lon=7.096829

In [18]:
for node in result.nodes:
    print(f'Node ID: {node.id}, Lat: {node.lat}, Lon: {node.lon}, Tags: {node.tags}')

Node ID: 31819742, Lat: 50.7317635, Lon: 7.0993974, Tags: {'VRS:gemeinde': 'BONN', 'VRS:name': 'Bonn Hauptbahnhof', 'VRS:ortsteil': 'Innenstadt', 'VRS:ref': '61101', 'bus': 'yes', 'local_ref': 'D2', 'name': 'Bonn Hauptbahnhof', 'public_transport': 'stop_position', 'ref': 'D2'}
Node ID: 86123030, Lat: 50.7320025, Lon: 7.0967372, Tags: {'VRS:gemeinde': 'BONN', 'VRS:name': 'Bonn Hauptbahnhof', 'VRS:ortsteil': 'Innenstadt', 'VRS:ref': '61101', 'local_ref': '3', 'name': 'Bonn Hauptbahnhof', 'network': 'VRS', 'official_name': 'Bonn Hbf', 'operator': 'DB Station&Service AG', 'public_transport': 'stop_position', 'railway': 'stop', 'railway:ref': 'KB', 'ref': '3', 'train': 'yes'}
Node ID: 253075334, Lat: 50.7325615, Lon: 7.0986046, Tags: {'addr:city': 'Bonn', 'addr:country': 'DE', 'addr:housenumber': '2-4', 'addr:postcode': '53111', 'addr:street': 'Gangolfstraße', 'brand': 'Denns BioMarkt', 'brand:wikidata': 'Q48883773', 'name': 'Denns BioMarkt', 'opening_hours': 'Mo-Sa 08:00-20:00', 'operator'

In [ ]:
# Initizialize the map centered around the specified location
m = folium.Map(location=[lat, lon], zoom_start=15, tiles='cartodb positron')

# MarkerCluster for better visualization of multiple points
marker_cluster = MarkerCluster().add_to(m)


# Helper function for categorizing nodes based on tags
def get_info(tags):
    if 'public_transport' in tags:
        return 'blue', 'ÖPNV'
    if 'shop' in tags:
        return 'green', 'Shopping'
    if tags.get('amenity') in ['cafe', 'restaurant']:
        return 'orange', 'Gastro'
    if tags.get('amenity') in ['school', 'university', 'arts_centre']:
        return 'purple', 'Bildung'
    return 'gray', 'Sonstiges'


# Show nodes on map
for node in result.nodes:
    color, cat = get_info(node.tags)
    name = node.tags.get('name', cat)

    folium.CircleMarker(
        location=[node.lat, node.lon],
        radius=6,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=f'{cat}: {name}',
    ).add_to(marker_cluster)

# Mark center point
folium.Marker(
    [lat, lon], popup='<b>SMLS Center</b>', icon=folium.Icon(color='red', icon='star')
).add_to(m)

# Draw radius circle
folium.Circle(
    location=[lat, lon], radius=radius, color='black', weight=1, fill=False
).add_to(m)

m.save('smls_map.html')
m